# ДЗ 1 — Побить линейную регрессию на House Prices

> Если зачем-то закрыли лекцию: это [Модуль 3](https://itrubnikov.github.io/Train_of_Thought/modules/03-catboost) курса «От нуля до своих агентов».

Цель — за вечер пройти полный пайплайн на классике Kaggle. Грузим **Ames Housing** (датасет [House Prices — Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)): ~1500 домов в городе Эймс, Айова, 80 фичей (год постройки, район, материал крыши, площадь подвала и так далее), нужно предсказать цену продажи. Это ровно та задача, которую мы разбирали в ментальной модели лекции: «дерево №1 говорит 10 млн, дерево №2 чинит ошибку…».

Обучаем линейную регрессию-бейзлайн, обгоняем её CatBoost'ом **без feature engineering'а**, смотрим SHAP-объяснения.

**Что от вас требуется:** заполнить 3 блока `TODO`. Остальное уже написано. CatBoost и SHAP установятся первой ячейкой.

**Время:** 45—60 минут.

**Как сдавать:** `Файл → Сохранить копию на Диске` → дописать `TODO` → запустить все ячейки → `Поделиться → у кого есть ссылка → Просмотр` → прислать ссылку в чат курса как `[Модуль 3, ДЗ 1] {ссылка}`.

## Шаг 0. Установка библиотек

В Colab нужны только два пакета сверху (`pandas`, `scikit-learn`, `matplotlib` уже стоят).

In [ ]:
!pip install -q catboost shap

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Шаг 1. Данные — House Prices (Ames Housing)

Самая ходовая Kaggle-классика для табличного ML: 1 460 домов в городе Эймс, штат Айова, продажи 2006—2010 годов. 80 признаков на дом — от площади гаража до качества кухни. Таргет — цена продажи `SalePrice` в долларах.

Грузим напрямую через `sklearn.datasets.fetch_openml` — ни Kaggle API, ни CSV качать не нужно. Первый запуск займёт ~10 секунд, дальше кэшируется.

In [ ]:
ames = fetch_openml(name='house_prices', as_frame=True, parser='auto')
df = ames.frame.copy()

# Лёгкая чистка: убираем технический id, оставляем всё остальное
df = df.drop(columns=['Id'])

print(df.shape)
df[['LotArea', 'YearBuilt', 'OverallQual', 'Neighborhood', 'GrLivArea', 'SalePrice']].head()

In [ ]:
# Таргет — log(SalePrice). Цены ходят от $35k до $755k, разброс на порядок —
# на лог-шкале RMSE интерпретируется как «относительная ошибка», и так
# меряют этот датасет на самой Kaggle-лидерборде.
y = np.log1p(df.pop('SalePrice'))
X = df

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(exclude='object').columns.tolist()
print(f'категориальных: {len(cat_features)}')
print(f'числовых: {len(num_features)}')
print(f'таргет log(SalePrice): mean={y.mean():.2f}, std={y.std():.2f}')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## Шаг 2. Бейзлайн — линейная регрессия (Ridge) с one-hot

Готовый код, просто запустите. Это та модель, которую вам нужно обогнать.

Два слова про **что это вообще такое**:

- **Линейная регрессия** — самая простая обучаемая модель для предсказания числа. Берёт все ваши признаки (площадь, год постройки, район…), умножает каждый на свой коэффициент-вес, складывает и выдаёт предсказание. Учится за секунды, объясняется в одну строку, и поэтому это **дефолтный бейзлайн** в табличном ML: если ваша новая навороченная модель не бьёт линейку — она вам не нужна. `Ridge` — это та же линейная регрессия, но с L2-регуляризацией, чтобы не сходила с ума на скоррелированных колонках и пропущенных значениях.
- **One-hot encoding** — способ скормить *текстовую* категорию в модель, которая умеет только числа. Колонка `Neighborhood` со значениями `NAmes / OldTown / CollgCr / ...` (25 районов) превращается в 25 отдельных числовых колонок, в каждой 0 или 1 («это NAmes? — да/нет»). Без этого `Ridge` не знает, что делать со строкой `'NAmes'`. У этого подхода есть боли (взрыв размерности на колонках с тысячами уникальных значений, плохая работа с редкими категориями) — про них в [лекции модуля](https://itrubnikov.github.io/Train_of_Thought/modules/03-catboost/). Главная фишка CatBoost'а — что ему всё это руками делать не надо.
- **`SimpleImputer` (опускаем для краткости)** + **`StandardScaler`** — приводят числовые колонки к нулевому среднему и единичному разбросу. Линейным моделям это важно, чтобы коэффициенты для «площади в кв.футах» и «года постройки» жили в одном масштабе. Деревьям (CatBoost и компании) — пофиг, они работают порогами и масштабу безразличны.
- **`ColumnTransformer` + `Pipeline`** — стандартный sklearn-приём, чтобы препроцессинг и сама модель ехали как одно целое: `.fit()` сначала обучает преобразования (one-hot, scaler), потом модель; `.predict()` применяет их в том же порядке. Страховка от типичного бага «забыл прогнать StandardScaler перед predict — получил мусор».
- **RMSE на log(SalePrice)** — метрика, которую мы меряем. На лог-цене ошибка ~0.13 означает «модель промахивается примерно на ±14% от цены». Это та же метрика, что используется на Kaggle-лидерборде этого датасета (RMSLE). 0.16—0.18 — приличный честный бейзлайн от линейки.

In [ ]:
# Чиним пропуски: для числовых — медиана, для категорий — отдельная метка 'missing'
X_tr_fill = X_tr.copy()
X_te_fill = X_te.copy()
for c in num_features:
    med = X_tr_fill[c].median()
    X_tr_fill[c] = X_tr_fill[c].fillna(med)
    X_te_fill[c] = X_te_fill[c].fillna(med)
for c in cat_features:
    X_tr_fill[c] = X_tr_fill[c].fillna('missing').astype(str)
    X_te_fill[c] = X_te_fill[c].fillna('missing').astype(str)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features),
])
ridge = Pipeline([
    ('prep', preprocess),
    ('reg', Ridge(alpha=10.0, random_state=RANDOM_STATE)),
])

t0 = time.time()
ridge.fit(X_tr_fill, y_tr)
ridge_time = time.time() - t0

ridge_rmse = mean_squared_error(y_te, ridge.predict(X_te_fill)) ** 0.5
print(f'Ridge  RMSE = {ridge_rmse:.4f}  ({ridge_time:.1f} s)')

## Шаг 3. TODO 1 — CatBoost

Обучите `CatBoostRegressor` так, чтобы:
- получить RMSE ≤ 0.135 на тесте (**меньше** = лучше, в отличие от AUC),
- передать `cat_features` ЯВНО (иначе будет беда),
- использовать `eval_set=(X_te, y_te)` и `early_stopping_rounds=50`,
- замерить время обучения в `cat_time`.

Обратите внимание: CatBoost умеет в пропущенные значения сам — никаких `fillna` для него не нужно. Передаём **сырые** `X_tr` и `X_te`, не "_fill" версии.

Подсказка-каркас:

```python
from catboost import CatBoostRegressor

# CatBoost не любит NaN внутри категориальных колонок — заменим на 'missing'
X_tr_cb = X_tr.copy()
X_te_cb = X_te.copy()
for c in cat_features:
    X_tr_cb[c] = X_tr_cb[c].fillna('missing').astype(str)
    X_te_cb[c] = X_te_cb[c].fillna('missing').astype(str)

cat_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    cat_features=cat_features,
    eval_metric='RMSE',
    early_stopping_rounds=50,
    random_seed=RANDOM_STATE,
    verbose=0,
)
t0 = time.time()
cat_model.fit(X_tr_cb, y_tr, eval_set=(X_te_cb, y_te))
cat_time = time.time() - t0
cat_rmse = mean_squared_error(y_te, cat_model.predict(X_te_cb)) ** 0.5
```

Перепишите ячейку под себя и **проверьте**, что RMSE ≤ 0.135.

In [ ]:
# TODO 1: обучить CatBoostRegressor, сохранить в cat_model, cat_rmse, cat_time, X_te_cb
raise NotImplementedError('Реализуй TODO 1 — см. инструкцию выше.')

print(f'CatBoost RMSE = {cat_rmse:.4f}  ({cat_time:.1f} s)')
assert cat_rmse <= 0.135, f'RMSE {cat_rmse:.4f} выше 0.135 — проверь cat_features, iterations и early_stopping_rounds'

In [ ]:
comparison = pd.DataFrame([
    {'model': 'Ridge + OneHot', 'RMSE (log price)': round(ridge_rmse, 4), 'time, s': round(ridge_time, 2)},
    {'model': 'CatBoost',       'RMSE (log price)': round(cat_rmse, 4),   'time, s': round(cat_time, 2)},
])
comparison.sort_values('RMSE (log price)')

## Шаг 4. TODO 2 — SHAP summary plot

Постройте `shap.summary_plot` для CatBoost-модели. Сохраните картинку как `shap_summary.png`.

Что такое SHAP, в одну строку: способ для каждой фичи показать, **насколько и в какую сторону** она в среднем толкает предсказание модели. Для регрессии знак прямой: «+0.10 в log-цене ≈ +10% к стоимости дома».

Каркас:

```python
import shap
explainer = shap.TreeExplainer(cat_model)
shap_values = explainer.shap_values(X_te_cb)
shap.summary_plot(shap_values, X_te_cb, max_display=10, show=False)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()
```

После запуска ответьте одним предложением (в ячейке ниже): **«топ-3 фичи, которые сильнее всего двигают цену дома, — это …»**. Спойлер: скорее всего там окажутся `OverallQual` (общее качество), `GrLivArea` (жилая площадь) и что-то из района или года постройки.

In [ ]:
# TODO 2: SHAP summary plot для CatBoost-модели
raise NotImplementedError('Реализуй TODO 2 — см. инструкцию выше.')

**Ваш ответ:** топ-3 фичи, которые сильнее всего двигают цену дома, — это … _(допишите)_

## Шаг 5. TODO 3 — Waterfall plot для одного дома

`summary_plot` показал общую картину. Теперь зум на один конкретный дом: возьмите **самый дорогой** дом в тестовой выборке (тот, для которого модель предсказала максимальную цену), постройте для него waterfall plot, сохраните как `shap_waterfall.png` и опишите одной фразой, какие именно фичи задрали его цену.

Каркас:

```python
preds = cat_model.predict(X_te_cb)
idx = int(np.argmax(preds))  # самый дорогой по предсказанию
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_te_cb.iloc[idx],
        feature_names=X_te_cb.columns.tolist(),
    ),
    show=False,
)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'предсказанная цена: ${np.expm1(preds[idx]):,.0f}')
```

In [ ]:
# TODO 3: waterfall plot для самого дорогого дома
raise NotImplementedError('Реализуй TODO 3 — см. инструкцию выше.')

**Ваш ответ:** модель оценила этот дом так дорого, потому что … _(допишите одной фразой)_

## Что вы только что сделали

За один вечер вы прошли путь, который ещё 10 лет назад занимал у ML-инженера неделю: загрузили реальный Kaggle-датасет (классику House Prices), обучили две модели (бейзлайн Ridge и CatBoost), сравнили их по RMSE и времени, объяснили лучшую через SHAP — глобально и по одному конкретному дому. Это и есть «правая ветка карты ML» из [Модуля 2](https://itrubnikov.github.io/Train_of_Thought/modules/02-ml-map) в действии.

**Чек перед сдачей:**
- [ ] CatBoost RMSE ≤ 0.135 (на log-цене).
- [ ] В таблице сравнения видны цифры обеих моделей.
- [ ] `shap_summary.png` сохранился и видно осмысленные фичи (`OverallQual`, `GrLivArea`, `Neighborhood`, `YearBuilt` — а не `Id`).
- [ ] `shap_waterfall.png` сохранился и подписан одной фразой.
- [ ] Прислана ссылка в чат курса как `[Модуль 3, ДЗ 1] {ссылка}`.

Дальше — [Модуль 4a: micrograd за час](https://itrubnikov.github.io/Train_of_Thought/modules/04a-micrograd).